# Demo Test Local

Bu notebook, Colab bagimliligi olmadan lokal Windows/conda ortaminda v5 DINOv2 + FAISS + LightGlue demo testi yapmak icindir.

Beklenen artifact'ler:
- `dinov2_kirsehir_v5_vitb14_cls_domainaug_best.pth`
- `faiss_index_v5_all.bin`
- `db_meta_v5_all.json`

Varsayilan olarak bu dosyalari proje kokundeki `Kirsehir_VPR_DINOv2_v5_Final/` klasorunde arar. Gerekirse ilk kod hucresindeki path'leri degistir.

In [1]:
# ==============================================================================
# HUCRE 1: LOKAL SISTEM HAZIRLIGI VE MODEL/VERITABANI YUKLEME
# ==============================================================================
# Eksik paket olursa conda env icinde su komutlari bir kere calistirabilirsin:
#!pip install faiss-cpu folium pillow-heif scikit-learn -q
#!pip install git+https://github.com/cvg/LightGlue.git -q

import os, json, shutil
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageOps

import torch
import torch.nn as nn
import torchvision.transforms as T
from IPython.display import display, HTML

try:
    import faiss
except ImportError as e:
    raise ImportError("faiss eksik. Conda env icinde: pip install faiss-cpu") from e

try:
    import folium
except ImportError as e:
    raise ImportError("folium eksik. Conda env icinde: pip install folium") from e

try:
    from pillow_heif import register_heif_opener
    register_heif_opener()
    print("HEIC destegi aktif.")
except Exception as e:
    print(f"HEIC destegi etkinlesmedi: {e}")

try:
    from lightglue import LightGlue, ALIKED, viz2d
    from lightglue.utils import load_image as _lg_load_image_orig, rbd
except ImportError as e:
    raise ImportError("LightGlue eksik. Conda env icinde: pip install git+https://github.com/cvg/LightGlue.git") from e

# -------------------------------
# LOKAL PATH AYARLARI
# -------------------------------
PROJECT_ROOT = Path.cwd()

# v5 egitim ciktilarini lokal bilgisayarda bu klasore koy:
ARTIFACT_DIR = PROJECT_ROOT / "Kirsehir_VPR_DINOv2_v5_Final"

# Veri setini lokal acarsan buraya koy. Sadece eslesen DB gorsellerini ekranda gostermek icin lazim.
# db_meta icindeki /content/kirsehir_data path'leri lokal path'e otomatik cevrilir.
LOCAL_ROOT = PROJECT_ROOT / "kirsehir_data"

# Test etmek istedigin telefon/harici foto yolunu buraya yaz.
# Ornek: QUERY_PATH = PROJECT_ROOT / "Kirsehir_VPR_CrossSource_Photos" / "01_545 Sokak" / "IMG_3128.HEIC"
QUERY_PATH = PROJECT_ROOT / "test.png"

MODEL_NAME = "dinov2_vitb14"
MODEL_POOLING = "cls"
OUTFEATURES = 512
IMG_SIZE = 518
RERANK_TOP_K = 20
LIGHTGLUE_MAX_SIDE = 1024

MODEL_PATH = ARTIFACT_DIR / "dinov2_kirsehir_v5_vitb14_cls_domainaug_best.pth"
FAISS_PATH = ARTIFACT_DIR / "faiss_index_v5_all.bin"
META_PATH = ARTIFACT_DIR / "db_meta_v5_all.json"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RESAMPLE_LANCZOS = getattr(Image, "Resampling", Image).LANCZOS

print(f"Proje klasoru : {PROJECT_ROOT}")
print(f"Artifact klasoru: {ARTIFACT_DIR}")
print(f"Cihaz: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

for p in [MODEL_PATH, FAISS_PATH, META_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"Gerekli dosya bulunamadi: {p}")

def open_rgb_image(path):
    img = Image.open(path)
    img = ImageOps.exif_transpose(img)
    return img.convert("RGB")

def load_image(path):
    """LightGlue icin EXIF duzeltilmis ve buyuk kenari sinirlanmis tensor yukleyici."""
    try:
        img_pil = open_rgb_image(path)
        if max(img_pil.size) > LIGHTGLUE_MAX_SIDE:
            img_pil.thumbnail((LIGHTGLUE_MAX_SIDE, LIGHTGLUE_MAX_SIDE), RESAMPLE_LANCZOS)
        img_np = np.array(img_pil)
        return torch.from_numpy(img_np).permute(2, 0, 1).float() / 255.0
    except Exception:
        return _lg_load_image_orig(str(path))

def localize_db_path(path_str):
    """Colab metadata path'ini lokal veri seti path'ine cevirir."""
    p = Path(path_str)
    if p.exists():
        return str(p)
    s = str(path_str).replace("\\", "/")
    marker = "/content/kirsehir_data/"
    if marker in s:
        rel = s.split(marker, 1)[1]
        cand = LOCAL_ROOT / Path(rel)
        if cand.exists():
            return str(cand)
        # Zip bir ust klasorle acildiyse ikinci seviye de denenir.
        if LOCAL_ROOT.exists():
            parts = rel.split("/", 1)
            for child in LOCAL_ROOT.iterdir():
                cand2 = child / Path(rel)
                if child.is_dir() and cand2.exists():
                    return str(cand2)
    return str(path_str)

class GeMPooling(nn.Module):
    def __init__(self, p=3, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p)
        self.eps = eps

    def forward(self, x):
        return x.clamp(min=self.eps).pow(self.p).mean(dim=1).pow(1.0 / self.p)

class VPRDINOv2(nn.Module):
    def __init__(self, backbone_name=MODEL_NAME, out_dim=OUTFEATURES, pooling=MODEL_POOLING):
        super().__init__()
        self.pooling = pooling.lower()
        self.backbone = torch.hub.load("facebookresearch/dinov2", backbone_name, verbose=False)
        embed_dim = self.backbone.embed_dim

        if self.pooling == "gem":
            self.gem = GeMPooling(p=3)
        elif self.pooling != "cls":
            raise ValueError(f"Desteklenmeyen pooling: {pooling}")

        self.head = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.BatchNorm1d(embed_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.10),
            nn.Linear(embed_dim, out_dim),
        )

    def forward(self, x):
        ret = self.backbone.forward_features(x)
        if self.pooling == "cls":
            x = ret["x_norm_clstoken"]
        else:
            x = self.gem(ret["x_norm_patchtokens"])
        x = self.head(x)
        return nn.functional.normalize(x, p=2, dim=1)

dino_model = VPRDINOv2(MODEL_NAME, OUTFEATURES, MODEL_POOLING).to(DEVICE)
state = torch.load(MODEL_PATH, map_location=DEVICE)
if isinstance(state, dict) and "model_state" in state:
    state = state["model_state"]
dino_model.load_state_dict(state, strict=True)
dino_model.eval()

phone_query_transform = T.Compose([
    T.Resize(IMG_SIZE, interpolation=T.InterpolationMode.BICUBIC),
    T.CenterCrop((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

lg_extractor = ALIKED(max_num_keypoints=1024).eval().to(DEVICE)
lg_matcher = LightGlue(features="aliked").eval().to(DEVICE)

faiss_index = faiss.read_index(str(FAISS_PATH))
with open(META_PATH, "r", encoding="utf-8") as f:
    db_meta = json.load(f)
for item in db_meta:
    item["filepath_local"] = localize_db_path(item["filepath"])

GLOBAL_QUERY_PATH = None
GLOBAL_DINO_TOP20 = []
GLOBAL_BEST_HYBRID = None
GLOBAL_RERANKED_RESULTS = []

print("Model, FAISS ve metadata hazir.")
print(f"FAISS vektor sayisi: {faiss_index.ntotal}")

HEIC destegi aktif.
Proje klasoru : c:\Users\Admin\Desktop\proje
Artifact klasoru: c:\Users\Admin\Desktop\proje\Kirsehir_VPR_DINOv2_v5_Final
Cihaz: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to C:\Users\Admin/.cache\torch\hub\main.zip
C:\Users\Admin/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
C:\Users\Admin/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
C:\Users\Admin/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vitb14/dinov2_vitb14_pretrain.pth" to C:\Users\Admin/.cache\torch\hub\checkpoints\dinov2_vitb14_pretrain.pth
7.8%


KeyboardInterrupt: 

In [ ]:
# ==============================================================================
# HUCRE 2: SORGU FOTOGRAFI SEC VE DINOv2 GLOBAL ARAMA
# ==============================================================================
# QUERY_PATH'i Hucre 1'de ayarlamadiysan burada elle yazabilirsin:
# QUERY_PATH = PROJECT_ROOT / "Kirsehir_VPR_CrossSource_Photos" / "01_545 Sokak" / "IMG_3128.HEIC"

if QUERY_PATH is None:
    raise ValueError("Once QUERY_PATH degiskenine test fotografi yolunu yaz.")

GLOBAL_QUERY_PATH = Path(QUERY_PATH)
if not GLOBAL_QUERY_PATH.exists():
    raise FileNotFoundError(f"Sorgu fotografi bulunamadi: {GLOBAL_QUERY_PATH}")

img_pil = open_rgb_image(GLOBAL_QUERY_PATH)
img_tensor = phone_query_transform(img_pil).unsqueeze(0).to(DEVICE)

with torch.no_grad(), torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
    query_emb = dino_model(img_tensor).cpu().numpy().astype(np.float32)
faiss.normalize_L2(query_emb)

search_k = min(RERANK_TOP_K, faiss_index.ntotal)
dists, idxs = faiss_index.search(query_emb, search_k)

GLOBAL_DINO_TOP20 = []
for i in range(search_k):
    match = db_meta[int(idxs[0][i])].copy()
    match["dino_score"] = float(dists[0][i])
    match["db_idx"] = int(idxs[0][i])
    GLOBAL_DINO_TOP20.append(match)

display(HTML("<h3>DINOv2 v5 Sonuclari (Top-3)</h3>"))
n_show = min(3, len(GLOBAL_DINO_TOP20))
fig, axes = plt.subplots(1, n_show + 1, figsize=(4.5 * (n_show + 1), 4))
if n_show == 0:
    axes = [axes]
axes[0].imshow(img_pil)
axes[0].set_title("SORGU", color="blue", fontweight="bold")
axes[0].axis("off")

for i in range(n_show):
    p = GLOBAL_DINO_TOP20[i].get("filepath_local", GLOBAL_DINO_TOP20[i]["filepath"])
    try:
        m_img = open_rgb_image(p)
        axes[i + 1].imshow(m_img)
    except Exception as e:
        axes[i + 1].text(0.5, 0.5, f"Gorsel acilamadi\n{Path(p).name}", ha="center", va="center")
    axes[i + 1].set_title(
        f"#{i+1}: {GLOBAL_DINO_TOP20[i]['street'][:24]}\nSkor: {GLOBAL_DINO_TOP20[i]['dino_score']:.3f}",
        fontweight="bold"
    )
    axes[i + 1].axis("off")
plt.tight_layout()
plt.show()

best_dino = GLOBAL_DINO_TOP20[0]
print(f"DINOv2 v5'e gore en olasi konum: {best_dino['street']}")
print(f"Koordinat: {best_dino['lat']:.6f}, {best_dino['lon']:.6f}")
print("\nTop-20 sokak adaylari:")
for i, m in enumerate(GLOBAL_DINO_TOP20, 1):
    print(f"{i:>2}. {m['street']:<35} skor={m['dino_score']:.3f}")

In [ ]:
# ==============================================================================
# HUCRE 3: LIGHTGLUE ILE TOP-20 RE-RANKING
# ==============================================================================
if not GLOBAL_DINO_TOP20:
    raise RuntimeError("Once Hucre 2'yi calistirip DINOv2 Top-20 adaylarini uret.")

print(f"DINOv2'nin buldugu {len(GLOBAL_DINO_TOP20)} aday LightGlue ile taraniyor...\n")

img0 = load_image(GLOBAL_QUERY_PATH).to(DEVICE)
with torch.inference_mode():
    feats0 = lg_extractor.extract(img0)

reranked_results = []
for rank, cand in enumerate(GLOBAL_DINO_TOP20):
    num_matches = 0
    cand_path = cand.get("filepath_local", cand["filepath"])
    try:
        img1 = load_image(cand_path).to(DEVICE)
        with torch.inference_mode():
            feats1 = lg_extractor.extract(img1)
            matches01 = lg_matcher({"image0": feats0, "image1": feats1})
        matches_clean = rbd(matches01)
        num_matches = int(matches_clean["matches"].shape[0])
        del img1, feats1, matches01, matches_clean
    except Exception as e:
        print(f"Aday #{rank + 1} LightGlue hatasi: {type(e).__name__}: {e}")

    cand_copy = cand.copy()
    cand_copy["num_matches"] = num_matches
    cand_copy["original_rank"] = rank + 1
    reranked_results.append(cand_copy)

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

del img0, feats0
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

GLOBAL_RERANKED_RESULTS = sorted(reranked_results, key=lambda x: x["num_matches"], reverse=True)
GLOBAL_BEST_HYBRID = GLOBAL_RERANKED_RESULTS[0]

display(HTML("<h3>LightGlue ile Yeniden Siralanmis Sonuclar (Top-3)</h3>"))
n_show = min(3, len(GLOBAL_RERANKED_RESULTS))
fig, axes = plt.subplots(1, n_show + 1, figsize=(4.5 * (n_show + 1), 4))
axes[0].imshow(open_rgb_image(GLOBAL_QUERY_PATH))
axes[0].set_title("SORGU", color="blue", fontweight="bold")
axes[0].axis("off")

for i in range(n_show):
    m = GLOBAL_RERANKED_RESULTS[i]
    p = m.get("filepath_local", m["filepath"])
    try:
        axes[i + 1].imshow(open_rgb_image(p))
    except Exception:
        axes[i + 1].text(0.5, 0.5, f"Gorsel acilamadi\n{Path(p).name}", ha="center", va="center")
    title_color = "red" if m["original_rank"] > 3 and i == 0 else "green"
    axes[i + 1].set_title(
        f"Yeni #{i+1}: {m['street'][:24]}\nEslesme: {m['num_matches']} (Eski: {m['original_rank']})",
        fontweight="bold",
        color=title_color
    )
    axes[i + 1].axis("off")
plt.tight_layout()
plt.show()

print(f"Hibrit sisteme gore konum: {GLOBAL_BEST_HYBRID['street']}")
print(f"Koordinat: {GLOBAL_BEST_HYBRID['lat']:.6f}, {GLOBAL_BEST_HYBRID['lon']:.6f}")
print("\nLightGlue siralamasi:")
for i, m in enumerate(GLOBAL_RERANKED_RESULTS, 1):
    print(f"{i:>2}. {m['street']:<35} eslesme={m['num_matches']:>4} eski_sira={m['original_rank']:>2} dino={m['dino_score']:.3f}")

m_map = folium.Map(location=[GLOBAL_BEST_HYBRID["lat"], GLOBAL_BEST_HYBRID["lon"]], zoom_start=17)
folium.Marker(
    [GLOBAL_BEST_HYBRID["lat"], GLOBAL_BEST_HYBRID["lon"]],
    popup=f"Tahmin: {GLOBAL_BEST_HYBRID['street']}",
    icon=folium.Icon(color="green", icon="ok-sign")
).add_to(m_map)
display(m_map)

In [ ]:
# ==============================================================================
# HUCRE 4: GORSEL KANIT - LIGHTGLUE ESLESMELERI
# ==============================================================================
if not GLOBAL_BEST_HYBRID:
    raise RuntimeError("Once Hucre 3'u calistir.")

print(f"Hibrit sistemin sectigi sokak: {GLOBAL_BEST_HYBRID['street']}\n")

img0 = load_image(GLOBAL_QUERY_PATH)
img1 = load_image(GLOBAL_BEST_HYBRID.get("filepath_local", GLOBAL_BEST_HYBRID["filepath"]))

img0_dev = img0.to(DEVICE)
img1_dev = img1.to(DEVICE)

with torch.inference_mode():
    feats0 = lg_extractor.extract(img0_dev)
    feats1 = lg_extractor.extract(img1_dev)
    matches01 = lg_matcher({"image0": feats0, "image1": feats1})

feats0_c, feats1_c, matches01_c = [rbd(x) for x in [feats0, feats1, matches01]]
kpts0, kpts1, matches = feats0_c["keypoints"], feats1_c["keypoints"], matches01_c["matches"]

if matches.shape[0] == 0:
    print("LightGlue eslesme bulamadi; gorsel kanit cizilemeyecek.")
else:
    m_kpts0, m_kpts1 = kpts0[matches[..., 0]], kpts1[matches[..., 1]]
    viz2d.plot_images([img0, img1])
    viz2d.plot_matches(m_kpts0, m_kpts1, color="lime", lw=1.5)
    viz2d.add_text(0, "SORGU", fs=14)
    viz2d.add_text(1, f"VERITABANI\n{GLOBAL_BEST_HYBRID['street']}\n({GLOBAL_BEST_HYBRID['num_matches']} eslesme)", fs=14)
    plt.show()